In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "false"

import jax
import jax.numpy as jnp
import optax

print("Backend:", jax.default_backend())
print("Devices:", jax.devices())
print("X64 habilitado:", jax.config.read("jax_enable_x64"))

Backend: gpu
Devices: [CudaDevice(id=0)]
X64 habilitado: False


In [ ]:
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import f1_score, recall_score, precision_score
from sklearn import metrics
from sklearn.preprocessing import MinMaxScaler

from experiments_params import generate_models_in_dictionary
from iqc_de import *
from iqc import IQC
from iqc_zhangetal import iqc_zhangetal, iqc_britoetal
from iqc_multdimensional import iqc_multidimensional

from sklearn.datasets import load_iris
from sklearn.datasets import load_wine
from sklearn.datasets import load_breast_cancer
import pickle
from ucimlrepo import fetch_ucirepo
import numpy as np

from open_datasets import load_pima_diabetes, load_caesarian_section
from experiments_params import generate_multiclass_datasets


data_bases = generate_multiclass_datasets()

results = {}

name_of_file = 'results_multiclass_with_regularization.pkl'

try:
    with open(name_of_file, 'rb') as f:
        results = pickle.load(f)
except FileNotFoundError:
    print("File not found, starting with an empty dictionary.")
    results = {}
    
for learning_rate in [0.1, 0.01, 0.001]:
    for RANDOM_STATE in [40,80,120,160,200,240,280,320,360,400]:  # Example random states
        for name, (X, y) in data_bases.items():
            print("BASE", name, "RANDOM_STATE", RANDOM_STATE)
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
            scaler = MinMaxScaler(feature_range=(0, 1))
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            number_of_features = X.shape[1]
            dicModels = generate_models_in_dictionary(number_of_features)


            for model_name, model_info in dicModels.items():
                
                for optimizer in ["optimizer"]:  #["optimizer", "pso"]:
                    if (name, model_name, optimizer, learning_rate, RANDOM_STATE) in results:
                        print(f"Skipping {name}, {model_name}, {optimizer}, {learning_rate}, {RANDOM_STATE} as it is already computed.")
                        continue


                    if model_name.startswith("iqc_multidimensional"):
                        N_e = int(model_name.split("_")[2])  # Extract the number after "iqc_multidimensional_"
                    elif model_name.startswith("iqc_de"):
                        N_e = next_power_of_two(number_of_features)
                    else:
                        N_e = 2  # Default value for other models

                    binary_iqc = IQC(
                        model_name=model_name,
                        iqc=model_info['iqc'],
                        number_of_params=model_info['number_of_params'],
                        method=optimizer,
                        n_particles=10,
                        max_steps=1000,
                        learning_rate=learning_rate,
                        verbose=False,
                        random_state=RANDOM_STATE,
                        N_e=N_e
                    )

                    multiclass_iqc = OneVsRestClassifier(
                        binary_iqc,
                        n_jobs=1,
                    )

                    multiclass_iqc.fit(X_train_scaled, y_train)

                    y_pred = multiclass_iqc.predict(X_test_scaled)

                    # Calculate metrics
                    accuracy = (y_pred == y_test).mean()
                    # For multi-class classification
                    f1_multi = f1_score(y_test, y_pred, average='macro') # or 'weighted', 'micro'
                    recall_multi = recall_score(y_test, y_pred, average='macro')
                    precision_multi = precision_score(y_test, y_pred, average='macro')
                    support_multi = classification_report(y_test, y_pred, output_dict=True)
                    
                    
                    best_classifiers_params = tuple()
                    for class_label, estimator in zip(multiclass_iqc.classes_,multiclass_iqc.estimators_):
                        #print("Classe:",class_label,"Loss:",estimator.best_loss_,"Parâmetros:",estimator.params_,)
                        best_classifiers_params+=(estimator.params_,)
                    
                    results[(name, model_name, optimizer, learning_rate, RANDOM_STATE)] = {
                                        'accuracy': accuracy,
                                        'f1_score': f1_multi,
                                        'recall': recall_multi,
                                        'precision': precision_multi,
                                        'best_params': best_classifiers_params,
                                    }

                    pickle.dump(results, open(name_of_file, 'wb'))


Arquivo ARFF encontrado: caesarian.csv.arff
Colunas originais: ['Age', 'Delivery number', 'Delivery time', 'Blood of Pressure', 'Heart Problem', 'Caesarian']
BASE iris RANDOM_STATE 40
Skipping iris, iqc_zhangetal, optimizer, 0.1, 40 as it is already computed.
Skipping iris, iqc_britoetal, optimizer, 0.1, 40 as it is already computed.
Skipping iris, iqc_alfa, optimizer, 0.1, 40 as it is already computed.
Skipping iris, iqc_multidimensional_2, optimizer, 0.1, 40 as it is already computed.
Skipping iris, iqc_multidimensional_4, optimizer, 0.1, 40 as it is already computed.
Skipping iris, iqc_multidimensional_8, optimizer, 0.1, 40 as it is already computed.
Skipping iris, iqc_de, optimizer, 0.1, 40 as it is already computed.
Skipping iris, iqc_de_multitarget_2, optimizer, 0.1, 40 as it is already computed.
Skipping iris, iqc_de_multitarget_3, optimizer, 0.1, 40 as it is already computed.
Skipping iris, iqc_multidimensional_2_multitarget_2, optimizer, 0.1, 40 as it is already computed.
Skip

/home/labfernandocin/Documents/UFPE/pesquisa/IQC_family/interactive-quantum-regressor-main/.IQCvenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/labfernandocin/Documents/UFPE/pesquisa/IQC_family/interactive-quantum-regressor-main/.IQCvenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/labfernandocin/Documents/UFPE/pesquisa/IQC_family/interactive-quantum-regressor-main/.IQCvenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedM

BASE caesarian_section RANDOM_STATE 280


/home/labfernandocin/Documents/UFPE/pesquisa/IQC_family/interactive-quantum-regressor-main/.IQCvenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/labfernandocin/Documents/UFPE/pesquisa/IQC_family/interactive-quantum-regressor-main/.IQCvenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/labfernandocin/Documents/UFPE/pesquisa/IQC_family/interactive-quantum-regressor-main/.IQCvenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedM

BASE iris RANDOM_STATE 320
BASE wine RANDOM_STATE 320
BASE pima_diabetes RANDOM_STATE 320


...

/home/labfernandocin/Documents/UFPE/pesquisa/IQC_family/interactive-quantum-regressor-main/.IQCvenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/labfernandocin/Documents/UFPE/pesquisa/IQC_family/interactive-quantum-regressor-main/.IQCvenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/labfernandocin/Documents/UFPE/pesquisa/IQC_family/interactive-quantum-regressor-main/.IQCvenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedM

BASE caesarian_section RANDOM_STATE 240


/home/labfernandocin/Documents/UFPE/pesquisa/IQC_family/interactive-quantum-regressor-main/.IQCvenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/labfernandocin/Documents/UFPE/pesquisa/IQC_family/interactive-quantum-regressor-main/.IQCvenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/labfernandocin/Documents/UFPE/pesquisa/IQC_family/interactive-quantum-regressor-main/.IQCvenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedM

BASE iris RANDOM_STATE 280
BASE wine RANDOM_STATE 280


/home/labfernandocin/Documents/UFPE/pesquisa/IQC_family/interactive-quantum-regressor-main/.IQCvenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/labfernandocin/Documents/UFPE/pesquisa/IQC_family/interactive-quantum-regressor-main/.IQCvenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/labfernandocin/Documents/UFPE/pesquisa/IQC_family/interactive-quantum-regressor-main/.IQCvenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedM